In [ ]:
import sys
import subprocess
import os
from pathlib import Path
import sqlite3
import pandas as pd
from time import perf_counter
from statistics import median

from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel


DB_PATH = Path("../data/ecommerce.sqlite")
STAGING = Path("spark_input")
PARQUET = Path("spark_input_parquet")
STAGING.mkdir(exist_ok=True)
PARQUET.mkdir(exist_ok=True)

TABLES = ["geolocation", "order_items", "products", "sellers", 
          "product_category_name_translation", "orders", "customers"]

if DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as con:
        for table in TABLES:
            out = STAGING / f"{table}.csv"
            first = True
            total = 0
            for chunk in pd.read_sql_query(f"SELECT * FROM {table}", con, chunksize=100_000):
                chunk.to_csv(out, index=False, mode="w" if first else "a", header=first, encoding="utf-8")
                total += len(chunk)
                first = False
            print(f"{table:40s} {total:>9d} rows")
else:
    print("Файл ecommerce.sqlite не найден! Положи его в папку с ноутбуком.")

geolocation                                1000163 rows
order_items                                 112650 rows
products                                     32951 rows
sellers                                       3095 rows
product_category_name_translation               71 rows
orders                                       99441 rows
customers                                    99441 rows


In [12]:
spark = (SparkSession.builder
    .master("local[*]")
    .appName("Practice4_SparkExecution")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .config("spark.driver.memory", "3g")
    .config("spark.executor.memory", "3g")
    .getOrCreate())


sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

Master: local[*]
Spark UI: http://postmusic451:4040


In [ ]:

geo = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "geolocation.csv"))
items = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "order_items.csv"))
products = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "products.csv"))
sellers = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "sellers.csv"))
translations = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "product_category_name_translation.csv"))
orders = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "orders.csv"))
customers = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "customers.csv"))

print("Данные успешно загружены из CSV!")

Данные успешно загружены из CSV!


In [14]:
sc.setJobGroup("P4_LAZY", "Lazy evaluation demo")

geo_prepared = (geo
    .select("geolocation_zip_code_prefix", "geolocation_state", "geolocation_lat", "geolocation_lng")
    .filter(F.col("geolocation_state").isNotNull())
    .withColumn("abs_lat", F.abs(F.col("geolocation_lat")))
)
print("До Action (count) Spark ничего не считает. Тип:", type(geo_prepared).__name__)

t0 = perf_counter()
n = geo_prepared.count()
print(f"Rows: {n}, Elapsed: {perf_counter() - t0:.3f} s")

До Action (count) Spark ничего не считает. Тип: DataFrame
Rows: 1000163, Elapsed: 0.574 s


In [ ]:
print("Входных партиций:", geo.rdd.getNumPartitions())


geo.groupBy("geolocation_state").count().orderBy(F.desc("count")).show(5)


skewed16 = geo.repartition(16, "geolocation_state")
repart4 = skewed16.repartition(4)
coal4 = skewed16.coalesce(4)

print("=== repartition(4) ===")
repart4.explain("formatted")

print("=== coalesce(4) ===")
coal4.explain("formatted")

def partition_profile(df, name):
    print(f"--- {name} ---")
    (df.select(F.spark_partition_id().alias("pid"))
       .groupBy("pid").count().orderBy("pid").show())

partition_profile(repart4, "repartition(4)")
partition_profile(coal4, "coalesce(4)")

Входных партиций: 16
+-----------------+------+
|geolocation_state| count|
+-----------------+------+
|               SP|404268|
|               MG|126336|
|               RJ|121169|
|               RS| 61851|
|               PR| 57859|
+-----------------+------+
only showing top 5 rows
=== repartition(4) ===
== Physical Plan ==
Exchange (2)
+- Scan csv  (1)


(1) Scan csv 
Output [6]: [geolocation_id#379, geolocation_zip_code_prefix#380, geolocation_lat#381, geolocation_lng#382, geolocation_city#383, geolocation_state#384]
Batched: false
Location: InMemoryFileIndex [file:/d:/Documents/Learning/7/ТИАБД/задания/4/spark_input/geolocation.csv]
ReadSchema: struct<geolocation_id:int,geolocation_zip_code_prefix:int,geolocation_lat:double,geolocation_lng:double,geolocation_city:string,geolocation_state:string>

(2) Exchange
Input [6]: [geolocation_id#379, geolocation_zip_code_prefix#380, geolocation_lat#381, geolocation_lng#382, geolocation_city#383, geolocation_state#384]
Arguments: RoundRob

In [16]:
sc.setJobGroup("P4_GROUPBY", "Shuffle: groupBy seller_id")

seller_stats = (items
    .groupBy("seller_id")
    .agg(F.count("*").alias("items"), F.round(F.sum("price"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
seller_stats.explain("formatted")
seller_stats.show(5)

== Physical Plan ==
* Sort (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- Exchange (3)
         +- * HashAggregate (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [2]: [seller_id#405, price#407]
Batched: false
Location: InMemoryFileIndex [file:/d:/Documents/Learning/7/ТИАБД/задания/4/spark_input/order_items.csv]
ReadSchema: struct<seller_id:string,price:double>

(2) HashAggregate [codegen id : 1]
Input [2]: [seller_id#405, price#407]
Keys [1]: [seller_id#405]
Functions [2]: [partial_count(1), partial_sum(price#407)]
Aggregate Attributes [2]: [count#598L, sum#599]
Results [3]: [seller_id#405, count#600L, sum#601]

(3) Exchange
Input [3]: [seller_id#405, count#600L, sum#601]
Arguments: hashpartitioning(seller_id#405, 8), ENSURE_REQUIREMENTS, [plan_id=1380]

(4) HashAggregate [codegen id : 2]
Input [3]: [seller_id#405, count#600L, sum#601]
Keys [1]: [seller_id#405]
Functions [2]: [count(1), sum(price#407)]
Aggregate Attributes [2]: [count(1)#595L, sum(price#407)#596]
Resul

In [ ]:
spark.catalog.clearCache()


joined = (items
    .join(products.select("product_id", "product_category_name"), "product_id", "left")
    .join(sellers.select("seller_id", "seller_state"), "seller_id", "left")
    .withColumn("gross_value", F.col("price") + F.col("freight_value"))
)

def action_1(df): 
    return df.groupBy("product_category_name").agg(F.sum("gross_value")).first()[0]

def action_2(df): 
    return df.groupBy("seller_state").agg(F.sum("gross_value")).first()[0]


sc.setJobGroup("P4_NOCACHE", "No cache")
t0 = perf_counter()
_ = action_1(joined)
t1 = perf_counter()
_ = action_2(joined)
t2 = perf_counter()
print(f"Без кэша: Action 1 {t1-t0:.3f}s | Action 2 {t2-t1:.3f}s")


cached = joined.persist(StorageLevel.MEMORY_AND_DISK)


t0 = perf_counter()
_ = cached.count() 
t1 = perf_counter() 


_ = action_1(cached)
t2 = perf_counter()
_ = action_2(cached)
t3 = perf_counter()

print(f"С кэшем: Материализация {t1-t0:.3f}s | A1 {t2-t1:.3f}s | A2 {t3-t2:.3f}s")

# Освобождаем кэш
cached.unpersist()

Без кэша: Action 1 1.202s | Action 2 0.960s
С кэшем: Материализация 1.580s | A1 0.529s | A2 0.240s


DataFrame[seller_id: string, product_id: string, order_id: string, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, seller_state: string, gross_value: double]

In [19]:
N = 2_000_000
P_VALUES = [1, 2, 4, 8, 16, 32]
spark.conf.set("spark.sql.shuffle.partitions", "8")

def run_benchmark(p, repeats=3):
    times = []
    for _ in range(repeats):
        df = (spark.range(0, N, 1, numPartitions=p)
              .withColumn("key", (F.col("id") % 1000).cast("int"))
              .withColumn("value", F.sqrt(F.col("id") + 1.0)))
        t0 = perf_counter()
        _ = df.groupBy("key").agg(F.sum("value").alias("s")).agg(F.sum("s")).first()[0]
        times.append(perf_counter() - t0)
    return times, median(times)

_ = run_benchmark(4, repeats=1) # Прогрев
base_time = None
for p in P_VALUES:
    times, med = run_benchmark(p, repeats=3)
    if base_time is None: base_time = med
    print(f"p={p:<3} | Runs: {[round(t,2) for t in times]} | Median: {med:.3f}s | Speedup: {base_time/med:.2f}x")

p=1   | Runs: [0.23, 0.09, 0.08] | Median: 0.094s | Speedup: 1.00x
p=2   | Runs: [0.27, 0.15, 0.14] | Median: 0.152s | Speedup: 0.62x
p=4   | Runs: [0.2, 0.18, 0.22] | Median: 0.204s | Speedup: 0.46x
p=8   | Runs: [0.31, 0.17, 0.25] | Median: 0.247s | Speedup: 0.38x
p=16  | Runs: [0.33, 0.25, 0.22] | Median: 0.248s | Speedup: 0.38x
p=32  | Runs: [0.44, 0.33, 0.32] | Median: 0.332s | Speedup: 0.28x


In [20]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "32")
adaptive = geo.groupBy("geolocation_state").count()
adaptive.explain("formatted")
_ = adaptive.count()
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [1]: [geolocation_state#384]
Batched: false
Location: InMemoryFileIndex [file:/d:/Documents/Learning/7/ТИАБД/задания/4/spark_input/geolocation.csv]
ReadSchema: struct<geolocation_state:string>

(2) HashAggregate
Input [1]: [geolocation_state#384]
Keys [1]: [geolocation_state#384]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1937L]
Results [2]: [geolocation_state#384, count#1938L]

(3) Exchange
Input [2]: [geolocation_state#384, count#1938L]
Arguments: hashpartitioning(geolocation_state#384, 32), ENSURE_REQUIREMENTS, [plan_id=3154]

(4) HashAggregate
Input [2]: [geolocation_state#384, count#1938L]
Keys [1]: [geolocation_state#384]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#1936L]
Results [2]: [geolocation_state#384, count(1)#1936L AS count#1929L]

(5) AdaptiveSparkPlan
Output [2]: [geolocati

In [21]:
print(f"Orders: {orders.count()} строк, партиций: {orders.rdd.getNumPartitions()}")

orders_prepared = (orders
    .filter(F.col("order_status") == "delivered")
    .select("order_id", "customer_id", "order_purchase_timestamp")
    .withColumn("year", F.year("order_purchase_timestamp"))
)
t0 = perf_counter(); n = orders_prepared.count()
print(f"Доставленных: {n}, время: {perf_counter() - t0:.3f} s")

orders.groupBy("order_status").count().orderBy(F.desc("count")).explain("formatted")

Orders: 99441 строк, партиций: 5
Доставленных: 96478, время: 0.227 s
== Physical Plan ==
* Sort (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- Exchange (3)
         +- * HashAggregate (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [1]: [order_status#494]
Batched: false
Location: InMemoryFileIndex [file:/d:/Documents/Learning/7/ТИАБД/задания/4/spark_input/orders.csv]
ReadSchema: struct<order_status:string>

(2) HashAggregate [codegen id : 1]
Input [1]: [order_status#494]
Keys [1]: [order_status#494]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1987L]
Results [2]: [order_status#494, count#1988L]

(3) Exchange
Input [2]: [order_status#494, count#1988L]
Arguments: hashpartitioning(order_status#494, 8), ENSURE_REQUIREMENTS, [plan_id=3325]

(4) HashAggregate [codegen id : 2]
Input [2]: [order_status#494, count#1988L]
Keys [1]: [order_status#494]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#1986L]
Results [2]: [order_status#494, count(1)

In [ ]:
smj = orders.hint("merge").join(customers.hint("merge"), on="customer_id", how="inner")
print("=== JOIN Explain ===")
smj.explain("formatted")
t0 = perf_counter(); print("Rows:", smj.count()); print(f"Time: {perf_counter() - t0:.3f} s")

print("\n[Примечание: Broadcast Hash Join пропущен из-за ошибки TaskResultLost (нехватка RAM на локальном стенде)]")

# Cache
spark.catalog.clearCache()

joined_self = orders.join(customers.select("customer_id", "customer_state"), "customer_id", "left")

def act1(df): return df.groupBy("order_status").count().first()[0]
def act2(df): return df.groupBy("customer_state").count().first()[0]

t0 = perf_counter(); _ = act1(joined_self); t1 = perf_counter(); _ = act2(joined_self); t2 = perf_counter()
print(f"\nБез кэша: A1 {t1-t0:.3f}s | A2 {t2-t1:.3f}s")

cached_self = joined_self.persist(StorageLevel.MEMORY_AND_DISK)
t0 = perf_counter(); _ = cached_self.count(); t1 = perf_counter()
_ = act1(cached_self); t2 = perf_counter(); _ = act2(cached_self); t3 = perf_counter()
print(f"С кэшем: Мат {t1-t0:.3f}s | A1 {t2-t1:.3f}s | A2 {t3-t2:.3f}s")
cached_self.unpersist()

=== JOIN Explain ===
== Physical Plan ==
* Project (10)
+- * SortMergeJoin Inner (9)
   :- * Sort (4)
   :  +- Exchange (3)
   :     +- * Filter (2)
   :        +- Scan csv  (1)
   +- * Sort (8)
      +- Exchange (7)
         +- * Filter (6)
            +- Scan csv  (5)


(1) Scan csv 
Output [8]: [order_id#492, customer_id#493, order_status#494, order_purchase_timestamp#495, order_approved_at#496, order_delivered_carrier_date#497, order_delivered_customer_date#498, order_estimated_delivery_date#499]
Batched: false
Location: InMemoryFileIndex [file:/d:/Documents/Learning/7/ТИАБД/задания/4/spark_input/orders.csv]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,order_approved_at:timestamp,order_delivered_carrier_date:timestamp,order_delivered_customer_date:timestamp,order_estimated_delivery_date:timestamp>

(2) Filter [codegen id : 1]
Input [8]: [order_id#492, customer_id#493, order_statu

DataFrame[customer_id: string, order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, customer_state: string]